# 01, Data Loading & Cleaning

**Dataset:** DataCo Supply Chain — 180 k order-item records, 53 raw columns.
**Output:** `data/clean.parquet`

In [1]:
import os

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from warnings import filterwarnings

filterwarnings('ignore')

In [2]:
plt.style.use('seaborn-v0_8-whitegrid')

# Remove global palette (optional)
# sns.set_palette('viridis')

# Define your own colors
primary_color   = '#4C72B0'
secondary_color = '#55A868'
accent_color    = '#C44E52'
danger_color    = '#8172B2'
neutral_color   = '#CCB974'

## 1. Load Raw Data

In [3]:
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1')
print('Raw shape:', df.shape)
df.head(3)

Raw shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


## 2. Standardizing Column Names

In [4]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace(r'[\s\(\)]+', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

df = df.rename(columns={
    'order_date_dateorders':       'order_date',
    'shipping_date_dateorders':    'shipping_date',
    'order_profit_per_order':      'order_profit',
    'days_for_shipping_real':      'days_shipping_real',
    'days_for_shipment_scheduled': 'days_shipment_scheduled',
})

print(df.columns.tolist())

['type', 'days_shipping_real', 'days_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date', 'shipping_mode']


## 3. Inspect Missing Values & Column Cardinality

In [5]:
print('Shape:', df.shape)
print('\nMissing values (top 10):')
print(df.isna().sum().sort_values(ascending=False).head(10))

Shape: (180519, 53)

Missing values (top 10):
product_description         180519
order_zipcode               155679
customer_lname                   8
customer_zipcode                 3
type                             0
order_profit                     0
order_item_cardprod_id           0
order_item_discount              0
order_item_discount_rate         0
order_item_id                    0
dtype: int64


In [6]:
for col in df.columns:
    if df[col].nunique() < 10:
        print(f'\n{col}:')
        print(df[col].value_counts())


type:
type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

days_shipping_real:
days_shipping_real
2    56618
3    28765
6    28723
4    28513
5    28163
0     5080
1     4657
Name: count, dtype: int64

days_shipment_scheduled:
days_shipment_scheduled
4    107752
2     35216
1     27814
0      9737
Name: count, dtype: int64

delivery_status:
delivery_status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

late_delivery_risk:
late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

customer_country:
customer_country
EE. UU.        111146
Puerto Rico     69373
Name: count, dtype: int64

customer_email:
customer_email
XXXXXXXXX    180519
Name: count, dtype: int64

customer_password:
customer_password
XXXXXXXXX    180519
Name: count, dtype: int64

customer_segment:
customer_segment
Consumer       93504
Corporate      54789
Home Office    32226


## 4. Drop Redundant / PII Columns

Remove:
- **PII** — customer name, email, password, address
- **ID surrogate keys** — not useful for analysis
- **Near-duplicates** — `benefit_per_order` mirrors `order_profit`
- **Zero-variance** — `product_status` has only one value
- **High-cardinality geo** — city/state/country replaced by `order_region`

In [7]:
columns_to_drop = [
    'product_description', 'product_image',
    'customer_email', 'customer_password',
    'customer_fname', 'customer_lname',
    'customer_street', 'customer_zipcode', 'order_zipcode',
    'longitude', 'latitude',
    'order_item_cardprod_id', 'order_item_id',
    'order_item_discount', 'order_item_discount_rate',
    'order_item_product_price', 'order_item_quantity',
    'category_id', 'department_id',
    'order_id', 'order_customer_id', 'customer_id',
    'product_card_id', 'product_category_id',
    'benefit_per_order',
    'product_status',
    'customer_city', 'order_city', 'order_country',
    'order_state', 'customer_state', 'market',
]

df = df.drop(columns=columns_to_drop)
print('Shape after dropping columns:', df.shape)

Shape after dropping columns: (180519, 21)


## 5. Filter & Type Conversion

In [8]:
df = df[df['delivery_status'] != 'Shipping canceled']

for col in ['order_date', 'shipping_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=False)

print('Cleaned shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False).head(5))

Cleaned shape: (172765, 21)

Missing values:
type                       0
order_item_profit_ratio    0
shipping_date              0
product_price              0
product_name               0
dtype: int64


In [9]:
print('Remaining columns:', df.columns.tolist())
df.head()

Remaining columns: ['type', 'days_shipping_real', 'days_shipment_scheduled', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_name', 'customer_country', 'customer_segment', 'department_name', 'order_date', 'order_item_profit_ratio', 'sales', 'order_item_total', 'order_profit', 'order_region', 'order_status', 'product_name', 'product_price', 'shipping_date', 'shipping_mode']


,type,days_shipping_real,days_shipment_scheduled,sales_per_customer,delivery_status,late_delivery_risk,category_name,customer_country,customer_segment,department_name,...,order_item_profit_ratio,sales,order_item_total,order_profit,order_region,order_status,product_name,product_price,shipping_date,shipping_mode
0,DEBIT,3,4,314.640015,Advance shipping,0,Sporting Goods,Puerto Rico,Consumer,Fitness,...,0.29,327.75,314.640015,91.250000,Southeast Asia,COMPLETE,Smart watch,327.75,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,311.359985,Late delivery,1,Sporting Goods,Puerto Rico,Consumer,Fitness,...,-0.80,327.75,311.359985,-249.089996,South Asia,PENDING,Smart watch,327.75,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,309.720001,Shipping on time,0,Sporting Goods,EE. UU.,Consumer,Fitness,...,-0.80,327.75,309.720001,-247.779999,South Asia,CLOSED,Smart watch,327.75,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,304.809998,Advance shipping,0,Sporting Goods,EE. UU.,Home Office,Fitness,...,0.08,327.75,304.809998,22.860001,Oceania,COMPLETE,Smart watch,327.75,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,298.250000,Advance shipping,0,Sporting Goods,Puerto Rico,Corporate,Fitness,...,0.45,327.75,298.250000,134.210007,Oceania,PENDING_PAYMENT,Smart watch,327.75,2018-01-15 11:24:00,Standard Class


## 6. Save Cleaned Data

In [10]:
os.makedirs('data', exist_ok=True)
df.to_parquet('data/clean.parquet', index=False)
print('Saved to data/clean.parquet')
print('Final shape:', df.shape)

Saved to data/clean.parquet
Final shape: (172765, 21)
